In [1]:
%pip install -q sentence-transformers faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import faiss
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer

In [3]:
def find_repository_root(start_path=Path.cwd()):
    for folder in [start_path, *start_path.parents]:
        if (folder / ".git").exists():
            return folder
    
    raise FileNotFoundError(
        "Could not locate the Git repository."
    )


REPO_ROOT = find_repository_root()
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"

chunks_path = (
    PROCESSED_DATA_DIR / "document_chunks.jsonl"
)

print("Repository:", REPO_ROOT)
print("Chunks:", chunks_path)

Repository: D:\analytics\A_Python_Code\creditlens-rag
Chunks: D:\analytics\A_Python_Code\creditlens-rag\data\processed\document_chunks.jsonl


In [4]:
chunks_df = pd.read_json(
    chunks_path,
    lines=True
)

print("Chunks loaded:", f"{len(chunks_df):,}")

chunks_df[
    [
        "chunk_id",
        "company",
        "reporting_year",
        "word_count"
    ]
].head()

Chunks loaded: 3,913


,chunk_id,company,reporting_year,word_count
0,F_2023_10K_chunk_0000,Ford,2023,180
1,F_2023_10K_chunk_0001,Ford,2023,180
2,F_2023_10K_chunk_0002,Ford,2023,180
3,F_2023_10K_chunk_0003,Ford,2023,180
4,F_2023_10K_chunk_0004,Ford,2023,180


In [5]:
print(chunks_df.loc[0, "text"][:1000])

f-20231231 FALSE 2023 FY 0000037996 http://fasb.org/us-gaap/2023#PropertyPlantAndEquipmentNet P3Y 10 4.5 http://fasb.org/us-gaap/2023#AccountsPayableAndAccruedLiabilitiesCurrent http://fasb.org/us-gaap/2023#AccountsPayableAndAccruedLiabilitiesCurrent P2Y P3Y 1 1 8 http://fasb.org/us-gaap/2023#NonoperatingIncomeExpense http://fasb.org/us-gaap/2023#NonoperatingIncomeExpense http://fasb.org/us-gaap/2023#NonoperatingIncomeExpense 0 http://fasb.org/us-gaap/2023#OtherAssets http://fasb.org/us-gaap/2023#RevenueFromContractWithCustomerExcludingAssessedTax http://fasb.org/us-gaap/2023#RevenueFromContractWithCustomerExcludingAssessedTax http://fasb.org/us-gaap/2023#RevenueFromContractWithCustomerExcludingAssessedTax http://fasb.org/us-gaap/2023#OtherLiabilitiesNoncurrent http://fasb.org/us-gaap/2023#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2023#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2023#OtherLiabilitiesCurrent http://fasb.org/us-gaap/2023#OtherLiabilitiesNoncurrent http://fasb.o

In [6]:
MODEL_NAME = "BAAI/bge-small-en-v1.5"

embedding_model = SentenceTransformer(
    MODEL_NAME,
    device="cpu"
)

print(
    "Embedding dimension:",
    embedding_model.get_sentence_embedding_dimension()
)

print(
    "Maximum sequence length:",
    embedding_model.max_seq_length
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
Maximum sequence length: 512


C:\Users\veln8\AppData\Local\Temp\ipykernel_23056\2513278748.py:10: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_model.get_sentence_embedding_dimension()


In [7]:
chunk_texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    chunk_texts,
    batch_size=16,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = embeddings.astype("float32")

Batches:   0%|          | 0/245 [00:00<?, ?it/s]

In [8]:
print("Embedding matrix shape:", embeddings.shape)
print("Data type:", embeddings.dtype)

Embedding matrix shape: (3913, 384)
Data type: float32


In [12]:
embedding_dimension = embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(
    embedding_dimension
)

faiss_index.add(embeddings)

print("Vectors indexed:", faiss_index.ntotal)

Vectors indexed: 3913


In [13]:
chunks_df["faiss_id"] = np.arange(
    len(chunks_df)
)

In [14]:
indexed_chunks_path = (
    PROCESSED_DATA_DIR / "indexed_chunks.jsonl"
)

chunks_df.to_json(
    indexed_chunks_path,
    orient="records",
    lines=True,
    force_ascii=False
)

In [15]:
indexed_chunks_path = (
    PROCESSED_DATA_DIR / "indexed_chunks.jsonl"
)

chunks_df.to_json(
    indexed_chunks_path,
    orient="records",
    lines=True,
    force_ascii=False
)

In [16]:
faiss_index_path = (
    PROCESSED_DATA_DIR /
    "creditlens_bge_small.faiss"
)

faiss.write_index(
    faiss_index,
    str(faiss_index_path)
)

print("Index saved to:", faiss_index_path)

Index saved to: D:\analytics\A_Python_Code\creditlens-rag\data\processed\creditlens_bge_small.faiss


In [17]:
QUERY_PREFIX = (
    "Represent this sentence for searching "
    "relevant passages: "
)

In [18]:
user_question = (
    "What factors could negatively affect "
    "Ford's liquidity?"
)

query_text = QUERY_PREFIX + user_question

query_embedding = embedding_model.encode(
    [query_text],
    normalize_embeddings=True,
    convert_to_numpy=True
).astype("float32")

In [19]:
TOP_K = 5

similarity_scores, result_ids = (
    faiss_index.search(
        query_embedding,
        TOP_K
    )
)

print("Result IDs:", result_ids[0])
print("Similarity scores:", similarity_scores[0])

Result IDs: [ 295 1441  309 1299 1454]
Similarity scores: [0.81719244 0.8165128  0.81261486 0.7883575  0.7878528 ]


In [20]:
search_results = (
    chunks_df
    .iloc[result_ids[0]]
    .copy()
    .reset_index(drop=True)
)

search_results.insert(
    0,
    "similarity_score",
    similarity_scores[0]
)

search_results[
    [
        "similarity_score",
        "company",
        "reporting_year",
        "chunk_id",
        "text"
    ]
]

,similarity_score,company,reporting_year,chunk_id,text
0,0.817192,Ford,2023,F_2023_10K_chunk_0295,"Ford Credit had $149 billion of assets, $68 bi..."
1,0.816513,Ford,2025,F_2025_10K_chunk_0315,the effects of regulatory changes on the finan...
2,0.812615,Ford,2023,F_2023_10K_chunk_0309,Inflationary pressure and fluctuations in comm...
3,0.788357,Ford,2025,F_2025_10K_chunk_0173,"cost of capital, impacting capital intensive b..."
4,0.787853,Ford,2025,F_2025_10K_chunk_0328,market value of Ford or Ford Credit’s investme...


In [22]:
for result_number, row in search_results.iterrows():
    print("=" * 80)
    print(f"RESULT {result_number + 1}")
    print(f"Score: {row['similarity_score']:.4f}")
    print(f"Company: {row['company']}")
    print(f"Year: {row['reporting_year']}")
    print(f"Chunk: {row['chunk_id']}")
    print()
    print(row["text"][:1200])
    print()

RESULT 1
Score: 0.8172
Company: Ford
Year: 2023
Chunk: F_2023_10K_chunk_0295

Ford Credit had $149 billion of assets, $68 billion of which were unencumbered. Funding and Liquidity Risks. Ford Credit’s funding plan is subject to risks and uncertainties, many of which are beyond its control, including disruption in the capital markets that could impact both unsecured debt and asset-backed securities issuance and the effects of regulatory changes on the financial markets. Despite Ford Credit’s diverse sources of funding and liquidity, its ability to maintain liquidity may be affected by, among others, the following factors (not necessarily listed in order of importance or probability of occurrence): • Prolonged disruption of the debt and securitization markets; • Global capital markets volatility; • Credit ratings assigned to Ford and Ford Credit; • Market capacity for Ford- and Ford Credit-sponsored investments; • General demand for the type of securities Ford Credit offers; • Ford Credi

In [23]:
def semantic_search(
    question,
    company=None,
    reporting_year=None,
    top_k=5
):
    """
    Search SEC filing chunks with optional
    company and reporting-year filters.
    """
    
    query_text = QUERY_PREFIX + question
    
    query_embedding = embedding_model.encode(
        [query_text],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).astype("float32")
    
    # Retrieve extra candidates before filtering
    candidate_count = min(
        max(top_k * 20, 100),
        faiss_index.ntotal
    )
    
    scores, ids = faiss_index.search(
        query_embedding,
        candidate_count
    )
    
    results = (
        chunks_df
        .iloc[ids[0]]
        .copy()
        .reset_index(drop=True)
    )
    
    results.insert(
        0,
        "similarity_score",
        scores[0]
    )
    
    if company is not None:
        results = results[
            results["company"].str.lower()
            == company.lower()
        ]
    
    if reporting_year is not None:
        results = results[
            results["reporting_year"].astype(str)
            == str(reporting_year)
        ]
    
    return (
        results
        .head(top_k)
        .reset_index(drop=True)
    )

In [24]:
ford_2025_results = semantic_search(
    question=(
        "What factors could negatively "
        "affect liquidity?"
    ),
    company="Ford",
    reporting_year=2025,
    top_k=5
)

In [25]:
ford_2025_results[
    [
        "similarity_score",
        "company",
        "reporting_year",
        "chunk_id"
    ]
]

,similarity_score,company,reporting_year,chunk_id
0,0.753011,Ford,2025,F_2025_10K_chunk_0315
1,0.738302,Ford,2025,F_2025_10K_chunk_0171
2,0.729594,Ford,2025,F_2025_10K_chunk_0356
3,0.728680,Ford,2025,F_2025_10K_chunk_0185
4,0.726562,Ford,2025,F_2025_10K_chunk_0430


In [26]:
for result_number, row in ford_2025_results.iterrows():
    print("=" * 80)
    print(f"RESULT {result_number + 1}")
    print(f"Score: {row['similarity_score']:.4f}")
    print(f"Company: {row['company']}")
    print(f"Year: {row['reporting_year']}")
    print(f"Chunk: {row['chunk_id']}")
    print()
    print(row["text"][:1200])
    print()

RESULT 1
Score: 0.7530
Company: Ford
Year: 2025
Chunk: F_2025_10K_chunk_0315

the effects of regulatory changes on the financial markets. Despite Ford Credit’s diverse sources of funding and liquidity, its ability to maintain liquidity may be affected by, among others, the following factors (not necessarily listed in order of importance or probability of occurrence): • Prolonged disruption of the debt and securitization markets • Global capital markets volatility • Credit ratings assigned to Ford and Ford Credit • Market capacity for Ford- and Ford Credit-sponsored investments • General demand for the type of securities Ford Credit offers • Ford Credit’s ability to continue funding through asset-backed financing structures • Performance of the underlying assets within Ford Credit’s asset-backed financing structures • Inability to obtain hedging instruments • Accounting and regulatory changes • Ford Credit’s ability to maintain credit facilities and committed asset-backed facilities Str

In [27]:
tesla_results = semantic_search(
    question=(
        "What risks could reduce cash flow "
        "or increase financing requirements?"
    ),
    company="Tesla",
    reporting_year=2025,
    top_k=5
)

tesla_results[
    [
        "similarity_score",
        "company",
        "reporting_year",
        "chunk_id",
        "text"
    ]
]

,similarity_score,company,reporting_year,chunk_id,text
0,0.743432,Tesla,2025,TSLA_2025_10K_chunk_0114,may fluctuate substantially from period to per...
1,0.718502,Tesla,2025,TSLA_2025_10K_chunk_0096,and compelling alternative financing programs ...
2,0.715470,Tesla,2025,TSLA_2025_10K_chunk_0113,are required to maintain a certain amount of l...
3,0.712343,Tesla,2025,TSLA_2025_10K_chunk_0200,our near-term manufacturing operations decreas...
4,0.708285,Tesla,2025,TSLA_2025_10K_chunk_0117,existing indebtedness and any future indebtedn...
